In [32]:
import json
import pprint
import collections
from pathlib import Path
import pandas as pd

DATA_FILE = Path("../data/darpa/ta1-cadets-e3-official/ta1-cadets-e3-official.json")

SAMPLE_SIZE = 200

print(f"File size : {DATA_FILE.stat().st_size / 1e6:.1f} MB")

File size : 4306.1 MB


In [33]:
# Load a sample of records for exploration 
raw_records = []

with DATA_FILE.open("r", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        line = line.strip()
        if not line:
            continue
        try:
            raw_records.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"[WARN] Line {i} parse error: {e}")
        if len(raw_records) >= SAMPLE_SIZE:
            break

print(f"Loaded {len(raw_records)} records for exploration.")

Loaded 200 records for exploration.


In [34]:
pprint.pprint(raw_records[0], depth=4)

{'CDMVersion': '18',
 'datum': {'com.bbn.tc.schema.avro.cdm18.Host': {'hostIdentifiers': [],
                                                 'hostName': 'ta1-cadets',
                                                 'hostType': 'HOST_DESKTOP',
                                                 'interfaces': [{...}, {...}],
                                                 'osDetails': 'FreeBSD '
                                                              '12.0-CURRENT '
                                                              'FreeBSD '
                                                              '12.0-CURRENT #1 '
                                                              '1863588dca9(HEAD)-dirty: '
                                                              'Wed Feb 28 '
                                                              '17:23:37 UTC '
                                                              '2018     '
                                                     

In [35]:
#top-level keys 
top_keys = collections.Counter()
for r in raw_records:
    top_keys.update(r.keys())

print("Top-level keys:")
for k, v in top_keys.most_common():
    print(f"  {k:30s} → {v} records")

Top-level keys:
  datum                          → 200 records
  CDMVersion                     → 200 records
  source                         → 200 records


In [36]:
# CDM record types
type_counter = collections.Counter()

for r in raw_records:
    datum = r.get("datum", {})
    for record_type in datum.keys():
        type_counter[record_type] += 1

print("CDM record types in sample:")
for rtype, cnt in type_counter.most_common():
    print(f"  {rtype:60s} → {cnt}")

CDM record types in sample:
  com.bbn.tc.schema.avro.cdm18.Event                           → 158
  com.bbn.tc.schema.avro.cdm18.FileObject                      → 30
  com.bbn.tc.schema.avro.cdm18.Subject                         → 8
  com.bbn.tc.schema.avro.cdm18.Principal                       → 3
  com.bbn.tc.schema.avro.cdm18.Host                            → 1


In [37]:
# Loking into Event records
events = [
    r["datum"]["com.bbn.tc.schema.avro.cdm18.Event"]
    for r in raw_records
    if "com.bbn.tc.schema.avro.cdm18.Event" in r.get("datum", {})
]

print(f"Event records: {len(events)}")
print("\n── First Event record ──")
pprint.pprint(events[0])

Event records: 158

── First Event record ──
{'hostId': '83C8ED1F-5045-DBCD-B39F-918F0DF4F851',
 'location': None,
 'name': {'string': 'aue_close'},
 'parameters': {'array': []},
 'predicateObject': {'com.bbn.tc.schema.avro.cdm18.UUID': '42DD2C9E-36C2-11E8-BF66-D9AA8AFF4A69'},
 'predicateObject2': None,
 'predicateObject2Path': None,
 'predicateObjectPath': None,
 'programPoint': None,
 'properties': {'map': {'exec': 'python2.7',
                        'fd': '17',
                        'host': '83c8ed1f-5045-dbcd-b39f-918f0df4f851',
                        'ppid': '1',
                        'return_value': '0'}},
 'sequence': {'long': 0},
 'size': None,
 'subject': {'com.bbn.tc.schema.avro.cdm18.UUID': '72FB0406-3678-11E8-BF66-D9AA8AFF4A69'},
 'threadId': {'int': 100117},
 'timestampNanos': 1522706861813350340,
 'type': 'EVENT_CLOSE',
 'uuid': '3EDAE524-F140-566E-8E72-94FE35EDC809'}


In [38]:
# Event types
event_types = collections.Counter(e.get("type", "UNKNOWN") for e in events)

print("Event types in sample:")
for etype, cnt in event_types.most_common():
    print(f"  {etype:40s} → {cnt}")

Event types in sample:
  EVENT_MMAP                               → 56
  EVENT_CLOSE                              → 30
  EVENT_FCNTL                              → 20
  EVENT_OPEN                               → 20
  EVENT_READ                               → 12
  EVENT_WRITE                              → 6
  EVENT_CREATE_OBJECT                      → 3
  EVENT_EXIT                               → 3
  EVENT_FORK                               → 3
  EVENT_EXECUTE                            → 3
  EVENT_CONNECT                            → 1
  EVENT_ACCEPT                             → 1


In [39]:
# Event property keys and frequecy
prop_keys = collections.Counter()

for e in events:
    props = e.get("properties")
    if props and isinstance(props, dict):
        inner = props.get("map", {})
        if inner:
            prop_keys.update(inner.keys())

print(f"Property keys across {len(events)} events:")
for key, cnt in prop_keys.most_common():
    pct = 100 * cnt / len(events)
    print(f"  {key:30s} → {cnt:3d} / {len(events)}  ({pct:.0f}%)")

Property keys across 158 events:
  host                           → 158 / 158  (100%)
  exec                           → 158 / 158  (100%)
  ppid                           → 158 / 158  (100%)
  fd                             → 143 / 158  (91%)
  return_value                   →  84 / 158  (53%)
  partial_path                   →  74 / 158  (47%)
  arg_mem_flags                  →  56 / 158  (35%)
  ret_fd1                        →  24 / 158  (15%)
  ret_fd2                        →   3 / 158  (2%)
  arg_pid                        →   3 / 158  (2%)
  cmdLine                        →   3 / 158  (2%)
  address                        →   2 / 158  (1%)


In [40]:
for target_type in ["EVENT_EXECUTE", "EVENT_CONNECT"]:
    matches = [e for e in events if e.get("type") == target_type]
    if matches:
        print(f"\n── {target_type} ──")
        pprint.pprint(matches[0])


── EVENT_EXECUTE ──
{'hostId': '83C8ED1F-5045-DBCD-B39F-918F0DF4F851',
 'location': None,
 'name': {'string': 'aue_execve'},
 'parameters': {'array': []},
 'predicateObject': {'com.bbn.tc.schema.avro.cdm18.UUID': '330061CF-6BB3-C45E-B36B-B89D7EC4F443'},
 'predicateObject2': {'com.bbn.tc.schema.avro.cdm18.UUID': '7A350F75-9945-425D-8599-15CEBD426F06'},
 'predicateObject2Path': {'string': '/libexec/ld-elf.so.1'},
 'predicateObjectPath': {'string': '/usr/bin/vmstat'},
 'programPoint': None,
 'properties': {'map': {'cmdLine': '/usr/bin/vmstat -m',
                        'exec': 'bash',
                        'host': '83c8ed1f-5045-dbcd-b39f-918f0df4f851',
                        'ppid': '2549',
                        'return_value': '-1'}},
 'sequence': {'long': 40},
 'size': None,
 'subject': {'com.bbn.tc.schema.avro.cdm18.UUID': '4442CA52-36C2-11E8-BF66-D9AA8AFF4A69'},
 'threadId': {'int': 100214},
 'timestampNanos': 1522706863153351329,
 'type': 'EVENT_EXECUTE',
 'uuid': 'DBC29206-6

In [41]:
# WAZUH data
WAZUH_FILE = Path("../data/SIEM/WAZUH_alerts.json")

assert WAZUH_FILE.exists(), f"Not found: {WAZUH_FILE.resolve()}"
print(f"File size: {WAZUH_FILE.stat().st_size / 1e6:.1f} MB")

wazuh_records = []
with WAZUH_FILE.open("r", encoding="utf-8") as fh:
    for i, line in enumerate(fh):
        line = line.strip()
        if not line:
            continue
        try:
            wazuh_records.append(json.loads(line))
        except json.JSONDecodeError:
            pass
        if len(wazuh_records) >= 20:
            break

print(f"Loaded {len(wazuh_records)} Wazuh records")
print("\n── First Wazuh record ──")
pprint.pprint(wazuh_records[0])

File size: 0.3 MB
Loaded 20 Wazuh records

── First Wazuh record ──
{'agent': {'id': '000', 'name': 'amenadiel'},
 'decoder': {'name': 'systemd'},
 'full_log': '2026-04-27T22:43:30.865386+02:00 amenadiel systemd[1]: '
             'fwupd-refresh.service: Main process exited, code=exited, '
             'status=1/FAILURE',
 'id': '1777322611.0',
 'location': '/var/log/syslog',
 'manager': {'name': 'amenadiel'},
 'predecoder': {'program_name': 'systemd',
                'timestamp': '2026-04-27T22:43:30.865386+02:00'},
 'rule': {'description': 'Systemd: Service exited due to a failure.',
          'firedtimes': 1,
          'gdpr': ['IV_35.7.d'],
          'gpg13': ['4.3'],
          'groups': ['local', 'systemd'],
          'id': '40704',
          'level': 5,
          'mail': False},
 'timestamp': '2026-04-27T22:43:31.714+0200'}


In [42]:
def flatten_dict(d, parent_key="", sep="."):
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

# Collect all field paths across 20 records
wazuh_fields = collections.Counter()
for r in wazuh_records:
    wazuh_fields.update(flatten_dict(r).keys())

print("Wazuh fields (20 records):")
for field, cnt in wazuh_fields.most_common():
    pct = 100 * cnt / len(wazuh_records)
    print(f"  {field:45s} {cnt:2d}/20  ({pct:.0f}%)")

Wazuh fields (20 records):
  timestamp                                     20/20  (100%)
  rule.level                                    20/20  (100%)
  rule.description                              20/20  (100%)
  rule.id                                       20/20  (100%)
  rule.firedtimes                               20/20  (100%)
  rule.mail                                     20/20  (100%)
  rule.groups                                   20/20  (100%)
  agent.id                                      20/20  (100%)
  agent.name                                    20/20  (100%)
  manager.name                                  20/20  (100%)
  id                                            20/20  (100%)
  decoder.name                                  20/20  (100%)
  location                                      20/20  (100%)
  agent.ip                                      17/20  (85%)
  data.win.system.providerName                  17/20  (85%)
  data.win.system.providerGuid               

In [43]:
win_records   = [r for r in wazuh_records if "data" in r and "win" in r.get("data", {})]
linux_records = [r for r in wazuh_records if "predecoder" in r]

print("── Windows record ──")
pprint.pprint(win_records[0])

print("\n── Linux record ──")
pprint.pprint(linux_records[0])

── Windows record ──
{'agent': {'id': '001', 'ip': '192.168.126.129', 'name': 'Windows'},
 'data': {'win': {'eventdata': {'param1': 'Background Intelligent Transfer '
                                          'Service',
                                'param2': 'demand start',
                                'param3': 'auto start',
                                'param4': 'BITS'},
                  'system': {'channel': 'System',
                             'computer': 'DESKTOP-423JMCQ',
                             'eventID': '7040',
                             'eventRecordID': '1201',
                             'eventSourceName': 'Service Control Manager',
                             'keywords': '0x8080000000000000',
                             'level': '4',
                             'message': '"The start type of the Background '
                                        'Intelligent Transfer Service service '
                                        'was changed from demand 

##### DARPA TC and Wazuh SIEM Mappings

- Event.timestampNanos maps to timestamp
- Event.type maps to data.win.system.eventID
- Event.properties.map.exec maps to predecoder.program_name
- Event.properties.map.address maps to data.win.eventdata.ipAddress
- Subject.cid maps to data.win.system.processID
- Event.predicateObjectPath maps to data.win.eventdata.processName